# 3. Database Connections

Real bridge-data workflows at a DOT usually mean talking to a database rather
than reading a spreadsheet. This notebook shows the connection patterns
CivilPy uses to do that — opening a connection, running a query, and pulling
results into pandas — without exposing any private system.

The production apps this library supports (for example the internal
`snbi_ui` Django site) sit on a **PostgreSQL/PostGIS** database reached over an
SSH tunnel, and their credentials, table names, and sample rows are not public.
So instead of connecting to that, we **build our own practice database** from
ODOT's public [TIMS](https://tims.dot.state.oh.us/) bridge inventory and
query *that*. The SQL and the cursor mechanics are the same ones you would use
against the real Postgres database — only the connection line changes.

> **Note:** These notebooks are teaching material, not official ODOT
> publications. They use only publicly available data. To produce a
> branded PDF, see the export note at the end of the notebook.

[⬅️ 2. Data Visualization Libraries](2.%20Data%20Visualization%20Libraries.ipynb)
&nbsp;&nbsp;|&nbsp;&nbsp;
[4. Useful Tricks and Tools ➡️](4.%20Useful%20Tricks%20and%20Tools.ipynb)

## 1. The production pattern (for reference)

CivilPy talks to the real ODOT Postgres database through
`civilpy.general.database_tools.ssh_into_postgres`. It opens an SSH tunnel to
the database host, then connects with [`psycopg`](https://www.psycopg.org/) and
hands back a standard DB-API connection object.

The credentials below are placeholders — this cell is shown for reference and is
**not run here** (it needs private hosts and passwords you won't have). Notice
the shape of it, because everything after this section reuses the same
connect → cursor → execute → fetch lifecycle:

```python
from civilpy.general.database_tools import ssh_into_postgres

creds = {
    "SSH_HOST": "...", "SSH_PORT": 22,
    "SSH_USER": "...", "SSH_PASSWORD": "...",
    "DB_HOST": "127.0.0.1", "PORT": 5432,
    "PG_UN": "...", "PG_DB_PW": "...", "PG_DB_NAME": "...",
}
conn = ssh_into_postgres(creds)          # opens the tunnel + connects

with conn.cursor() as cur:               # DB-API cursor
    cur.execute("SELECT sfn, str_loc_carried FROM bridges_bridge LIMIT 5")
    rows = cur.fetchall()                # pull results
```

Installing the optional driver stack is done with the `db` extra:

```bash
pip install "civilpy[db]"
```

The rest of this notebook swaps that private Postgres connection for a local
SQLite file you can build yourself — so you can practice the exact same query
code with zero credentials.

## 2. Build the practice database from public TIMS data

This is the reproducible part: run the cells below and you'll end up with a
`tims_practice.db` SQLite file next to this notebook, populated with one
county's worth of real bridge records. Delete it and re-run any time to get a
clean environment.

We use `civilpy.state.ohio.DOT.TIMS.get_tims_data`, which paginates ODOT's
public Bridge Inventory REST service into a pandas DataFrame. Passing a `where`
clause keeps the pull small and fast — here, every bridge in Athens County
(`COUNTY_CD='ATH'`, ~550 rows). Swap in your own county code (or drop the
filter entirely for all ~45,000 Ohio bridges).

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

from civilpy.state.ohio.DOT.TIMS import (
    get_tims_data,
    NBI_MATERIAL_CODES,
    NBI_DESIGN_TYPE_CODES,
)

# Pull one county's bridges from the public TIMS service.
# Change the WHERE clause to pull a different county or the whole state.
raw = get_tims_data("Bridge", where="COUNTY_CD='ATH'")
raw.shape

Fetched 555 records. Total so far: 555


No more features found. Fetched 1000 total records.
DataFrame successfully created with shape: (555, 234)


(555, 234)

The raw layer has 200-plus NBI columns. For a readable practice table we keep a
handful of the ones bridge engineers reach for most, decode the NBI material and
design-type codes into words (using the dictionaries CivilPy ships), and turn
the epoch-millisecond `YR_BUILT` field into a plain calendar year.

In [2]:
keep = [
    "SFN", "STR_LOC_CARRIED", "COUNTY_CD", "DISTRICT", "YR_BUILT",
    "MAIN_STR_MTL_CD", "MAIN_STR_TYPE_CD", "DECK_AREA", "SUFF_RATING",
    "LATITUDE_DD", "LONGITUDE_DD",
]
bridges = raw[keep].copy()

# YR_BUILT arrives as epoch milliseconds -> convert to a 4-digit year.
bridges["YR_BUILT"] = pd.to_datetime(
    bridges["YR_BUILT"], unit="ms", errors="coerce"
).dt.year

# Decode NBI codes into human-readable descriptions.
bridges["MATERIAL"] = (
    bridges["MAIN_STR_MTL_CD"].astype(str).map(NBI_MATERIAL_CODES)
)
bridges["DESIGN_TYPE"] = (
    bridges["MAIN_STR_TYPE_CD"].astype(str).str.zfill(2).map(NBI_DESIGN_TYPE_CODES)
)

# Sufficiency rating is text in the service; make it numeric for sorting.
bridges["SUFF_RATING"] = pd.to_numeric(bridges["SUFF_RATING"], errors="coerce")

bridges.head()

,SFN,STR_LOC_CARRIED,COUNTY_CD,DISTRICT,YR_BUILT,MAIN_STR_MTL_CD,MAIN_STR_TYPE_CD,DECK_AREA,SUFF_RATING,LATITUDE_DD,LONGITUDE_DD,MATERIAL,DESIGN_TYPE
0,0500496,USR 33,ATH,10,2012,3,02,9495,98.4,39.468671,-82.226960,Steel,Stringer/Multi-beam or Girder
1,0500526,USR 33,ATH,10,1989,1,19,760,99.8,39.470987,-82.270680,Concrete,Culvert
2,0500550,W WASHINGTON ST,ATH,10,1992,5,05,1836,98.6,39.463863,-82.249496,Prestressed Concrete,Box Beam or Girders - Multiple
3,0500615,USR 33,ATH,10,2012,6,02,10350,99.3,39.433419,-82.198387,Prestressed Concrete Continuous,Stringer/Multi-beam or Girder
4,0500631,USR 33 WB,ATH,10,1960,4,02,18604,89.7,39.388273,-82.142238,Steel Continuous,Stringer/Multi-beam or Girder


Now write the DataFrame into a SQLite database. SQLite ships with Python (no
server, no install), so the whole "database" is just the single `tims_practice.db`
file. `DataFrame.to_sql` creates the `bridges` table and infers column types for
us; `if_exists="replace"` makes re-running this cell idempotent.

In [3]:
db_path = Path("tims_practice.db")
if db_path.exists():
    db_path.unlink()  # start clean so the notebook is fully reproducible

conn = sqlite3.connect(db_path)
bridges.to_sql("bridges", conn, index=False, if_exists="replace")

row_count = conn.execute("SELECT COUNT(*) FROM bridges").fetchone()[0]
print(f"Wrote {row_count} bridges to {db_path.resolve()}")

Wrote 555 bridges to /home/dane/projects/civilpy/Notebooks/tims_practice.db


A quick look at the schema SQLite inferred. This `bridges` table now stands in
for a table like `bridges_bridge` in the production Postgres database.

In [4]:
schema = pd.read_sql("PRAGMA table_info(bridges)", conn)
schema[["name", "type"]]

,name,type
0,SFN,TEXT
1,STR_LOC_CARRIED,TEXT
2,COUNTY_CD,TEXT
3,DISTRICT,TEXT
4,YR_BUILT,INTEGER
5,MAIN_STR_MTL_CD,TEXT
6,MAIN_STR_TYPE_CD,TEXT
7,DECK_AREA,INTEGER
8,SUFF_RATING,REAL
9,LATITUDE_DD,REAL


## 3. Connect and query the practice database

Everything below is ordinary DB-API code. If you point `conn` at the production
Postgres connection from Section 1 instead of this SQLite file, the same cursor
calls and (nearly) the same SQL run unchanged.

### The cursor pattern

Open a cursor, `execute` a **parameterized** query (always pass values as
parameters — never string-format them into SQL — to stay safe from injection),
then `fetchone`/`fetchall` the results.

In [5]:
target_sfn = "0500496"

cur = conn.execute(
    "SELECT SFN, STR_LOC_CARRIED, YR_BUILT, MATERIAL "
    "FROM bridges WHERE SFN = ?",
    (target_sfn,),
)
cur.fetchone()

('0500496', 'USR 33', 2012, 'Steel')

### Querying straight into pandas

For analysis it's usually easier to skip the cursor and let pandas run the query
with `read_sql`. Here are the five structurally-worst bridges in the county by
sufficiency rating — the kind of triage query an inspection program starts with.

In [6]:
worst = pd.read_sql(
    '''
    SELECT SFN, STR_LOC_CARRIED, YR_BUILT, MATERIAL, SUFF_RATING
    FROM bridges
    WHERE SUFF_RATING IS NOT NULL
    ORDER BY SUFF_RATING ASC
    LIMIT 5
    ''',
    conn,
)
worst

,SFN,STR_LOC_CARRIED,YR_BUILT,MATERIAL,SUFF_RATING
0,0549568,BLACKWOOD RD,1900,Wood or Timber,5.2
1,0501786,USR 50,1997,Concrete,11.0
2,0502146,USR 50,2013,Concrete,11.0
3,0501816,USR 50,1997,Concrete,11.0
4,0550205,T0657,1976,Steel,12.3


Aggregate queries work the same way. This one breaks the county's inventory down
by main-span material — a one-line `GROUP BY` that would look identical against
Postgres.

In [7]:
by_material = pd.read_sql(
    '''
    SELECT MATERIAL, COUNT(*) AS n
    FROM bridges
    GROUP BY MATERIAL
    ORDER BY n DESC
    ''',
    conn,
)
by_material

,MATERIAL,n
0,Concrete,229
1,Prestressed Concrete,116
2,Steel,85
3,Steel Continuous,69
4,Concrete Continuous,32
5,Prestressed Concrete Continuous,16
6,Wood or Timber,6
7,"Aluminum, Wrought Iron, or Cast Iron",2


Close the connection when you're done (or use `with sqlite3.connect(...) as conn:`
to have it managed for you).

In [8]:
conn.close()

## 4. From the practice DB to the real one

Because both SQLite and Postgres speak the Python DB-API, moving this code onto
the production `snbi_ui` database is almost entirely a matter of swapping the
connection line:

| Step                | Practice (this notebook)              | Production (`snbi_ui`)                          |
|---------------------|---------------------------------------|------------------------------------------------|
| Driver / install    | `sqlite3` (stdlib)                    | `psycopg` via `pip install "civilpy[db]"`      |
| Open connection     | `sqlite3.connect("tims_practice.db")` | `ssh_into_postgres(creds)`                     |
| Cursor + execute    | `conn.execute(sql, params)`           | `conn.cursor()` then `cur.execute(sql, params)`|
| Parameter marker    | `?`                                   | `%s`                                           |
| Read into pandas    | `pd.read_sql(sql, conn)`              | `pd.read_sql(sql, conn)`                        |

The SQL itself is standard enough that most `SELECT` queries port directly; the
main adjustments are the parameter placeholder (`?` vs `%s`) and real table and
column names. With that, you can read what the `snbi_ui` library is doing against
its Postgres backend and follow along here without ever touching private data.

## Exporting a branded PDF

To render this notebook as an ODOT-branded PDF, use CivilPy's converter — the
branding is applied only at export time, never baked into the notebook itself:

```python
from civilpy.general.jupyter import notebook_converter

notebook_converter("3. DB_connections.ipynb", branding="odot")
```

---

[⬅️ 2. Data Visualization Libraries](2.%20Data%20Visualization%20Libraries.ipynb)
&nbsp;&nbsp;|&nbsp;&nbsp;
[4. Useful Tricks and Tools ➡️](4.%20Useful%20Tricks%20and%20Tools.ipynb)